# Plot Checkpoint Reconstruction - v5

Select a v5 checkpoint and plot masked event-byte reconstruction quality for compact event chunks. v5 reconstructs only masked event tokens, so the heatmap shows target event bytes and absolute reconstruction error for masked events.

In [ ]:
from pathlib import Path

MODEL_VERSION = "v5"
WORKSPACE = None
SAMPLE_CACHE_ROOT = Path(r"D:\market-data\prepared\event_sample_cache")
BATCH_SIZE = 32
EVENTS_PER_CHUNK = 128
DEVICE = "cuda"
USE_AMP = True
CHECKPOINT_PATH = ""
RUNTIME_ROOT = Path(r"D:\TradingML\runtimes\masked_event_model\v5\pretrain")
ARCHITECTURE_OUTPUT_DIR = Path(r"D:\TradingML\runtimes\masked_event_model\v5\notebook_artifacts\plot_architecture")
PLOT_SAMPLE_INDEX = 0


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
from IPython.display import Image, Markdown, SVG, display


def resolve_workspace(explicit):
    if explicit:
        path = Path(explicit)
        if (path / "research" / "masked_event_model" / MODEL_VERSION).exists():
            return path
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates.extend([
        Path(r"D:\TradingCodes\quant-research-workbench"),
        Path(r"D:\TradingML\codes\masked_event_model\v5"),
        Path(r"\\DESKTOP-SAAI85T\Workstation-D\TradingML\codes\masked_event_model\v5"),
    ])
    for path in candidates:
        if (path / "research" / "masked_event_model" / MODEL_VERSION).exists():
            return path
    raise FileNotFoundError("Could not find a workspace root containing research/masked_event_model/v5.")

WORKSPACE = resolve_workspace(WORKSPACE)
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from research.masked_event_model.v5.config import LossConfig, MaskConfig, ModelConfig
from research.masked_event_model.v5.losses import masked_event_bce_loss, pack_bits
from research.masked_event_model.v5.masking import build_event_masks
from research.masked_event_model.v5.model import EVENT_BYTES, HEADER_BYTES, EventTokenMaskedAutoencoder
from research.mlops.event_sample_cache import EventSampleCacheDataConfig, discover_event_sample_shards, iter_event_sample_cache_epoch_batches, resolve_event_sample_cache_root

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (18, 8), "axes.titlesize": 14, "axes.labelsize": 11, "legend.fontsize": 10})
device = torch.device(DEVICE if DEVICE == "cpu" or torch.cuda.is_available() else "cpu")
cache_root = resolve_event_sample_cache_root(SAMPLE_CACHE_ROOT)
print("workspace:", WORKSPACE)
print("cache_root:", cache_root)
print("device:", device)


In [ ]:
def latest_checkpoint(runtime_root):
    candidates = list(runtime_root.glob("*/checkpoints/latest.pt")) + list(runtime_root.glob("*/checkpoints/checkpoint_latest.pt")) + list(runtime_root.glob("*/checkpoints/*.pt"))
    if not candidates:
        raise FileNotFoundError(f"No checkpoints found under {runtime_root}. Set CHECKPOINT_PATH manually.")
    return max(candidates, key=lambda path: path.stat().st_mtime)

checkpoint_path = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else latest_checkpoint(RUNTIME_ROOT)
checkpoint = torch.load(checkpoint_path, map_location="cpu")
checkpoint_config = checkpoint.get("config")
if not checkpoint_config:
    raise KeyError("Checkpoint does not contain a config payload.")
model_config = ModelConfig(**{k: v for k, v in checkpoint_config.get("model", {}).items() if k in ModelConfig.__dataclass_fields__})
mask_config = MaskConfig(**{k: v for k, v in checkpoint_config.get("masks", {}).items() if k in MaskConfig.__dataclass_fields__})
loss_config = LossConfig(**{k: v for k, v in checkpoint_config.get("losses", {}).items() if k in LossConfig.__dataclass_fields__})
model = EventTokenMaskedAutoencoder(events_per_chunk=EVENTS_PER_CHUNK, config=model_config).to(device)
model.load_state_dict(checkpoint.get("model", checkpoint.get("model_state_dict", checkpoint)), strict=True)
model.eval()
print("checkpoint:", checkpoint_path)
print("step:", checkpoint.get("step"))
print(f"model parameters={sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# Keras-style model summary and torchview graph.
# Optional packages for richer output:
#   pip install torchinfo torchview graphviz
# Graph rendering also needs the Graphviz dot executable on PATH.

ARCHITECTURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_header = torch.zeros((1, HEADER_BYTES), dtype=torch.uint8, device=device)
summary_events = torch.zeros((1, EVENTS_PER_CHUNK, EVENT_BYTES), dtype=torch.uint8, device=device)

class TorchInfoSummaryWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, header_uint8, events_uint8):
        masks = build_event_masks(events_uint8, mask_config)
        output = self.model(header_uint8, events_uint8, masks, mask_config=None)
        return torch.cat([output.event_bit_probs.flatten(), output.chunk_embedding.flatten()]).unsqueeze(0)

summary_model = TorchInfoSummaryWrapper(model).to(device)
try:
    from torchinfo import summary
    summary_text = str(summary(summary_model, input_data=(summary_header, summary_events), depth=8, col_names=("input_size", "output_size", "num_params", "trainable"), verbose=0))
except Exception as exc:
    summary_text = f"torchinfo summary failed: {exc}\n\n{model}"
summary_path = ARCHITECTURE_OUTPUT_DIR / f"masked_event_model_{MODEL_VERSION}_summary.txt"
summary_path.write_text(summary_text, encoding="utf-8")
display(Markdown("```text\n" + summary_text[:50000] + ("\n... truncated ..." if len(summary_text) > 50000 else "") + "\n```"))
try:
    from torchview import draw_graph
    graph = draw_graph(summary_model, input_data=(summary_header, summary_events), expand_nested=True, depth=5, save_graph=False)
    graph.visual_graph.attr(dpi="180")
    png_path = ARCHITECTURE_OUTPUT_DIR / f"masked_event_model_{MODEL_VERSION}_torchview.png"
    svg_path = ARCHITECTURE_OUTPUT_DIR / f"masked_event_model_{MODEL_VERSION}_torchview.svg"
    graph.visual_graph.render(filename=str(png_path.with_suffix("")), directory=str(png_path.parent), format="png", cleanup=True)
    graph.visual_graph.render(filename=str(svg_path.with_suffix("")), directory=str(svg_path.parent), format="svg", cleanup=True)
    if png_path.exists():
        display(Image(filename=str(png_path)))
except Exception as exc:
    print("torchview graph failed:", repr(exc))


In [ ]:
cache_config = EventSampleCacheDataConfig(
    cache_root=SAMPLE_CACHE_ROOT,
    split="train",
    batch_size=BATCH_SIZE,
    events_per_chunk=EVENTS_PER_CHUNK,
    seed=23,
    prefetch_shards=1,
    max_shards=1,
    shuffle_records=False,
    drop_last=False,
)
shards = discover_event_sample_shards(cache_config)
batch = next(iter_event_sample_cache_epoch_batches(cache_config, epoch=1, shards=shards))
print("batch size:", batch["header_uint8"].shape[0], "shards:", len(shards))
print("first shard:", shards[0].path)


In [ ]:
def move_tensor_batch(batch_dict, device):
    return {key: value.to(device, non_blocking=True) if torch.is_tensor(value) else value for key, value in batch_dict.items()}

device_batch = move_tensor_batch(batch, device)
masks = build_event_masks(device_batch["events_uint8"], mask_config)
with torch.inference_mode(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP and device.type == "cuda"):
    output = model(device_batch["header_uint8"], device_batch["events_uint8"], masks, mask_config=None)
    result = masked_event_bce_loss(output, loss_config, include_diagnostics=True)
print("loss:", float(result.loss.detach().cpu()))
print(json.dumps(result.metrics, indent=2, sort_keys=True))


In [ ]:
sample = int(PLOT_SAMPLE_INDEX)
event_target = batch["events_uint8"][sample].numpy().astype(float)
event_pred = np.full_like(event_target, np.nan, dtype=float)

masked_event_indices = output.masked_event_indices.detach().cpu()
event_probs = output.event_bit_probs.detach().cpu()
event_bytes = pack_bits(event_probs >= 0.5).numpy() if event_probs.numel() else np.array([])
for event_idx, pred_bytes in zip(masked_event_indices[sample].numpy(), event_bytes[sample]):
    event_pred[int(event_idx), :] = pred_bytes.astype(float)

event_error = np.abs(event_pred - event_target)
fig, axes = plt.subplots(3, 1, figsize=(18, 11), gridspec_kw={"height_ratios": [3, 3, 3]}, constrained_layout=True)
origin_ns = int(batch["origin_timestamp_ns"][sample])
fig.suptitle(f"v5 masked event reconstruction | sample={sample} | origin_ns={origin_ns}", fontweight="bold")
im0 = axes[0].imshow(event_target.T, aspect="auto", interpolation="nearest", cmap="viridis")
axes[0].set_title("target event bytes")
axes[0].set_ylabel("byte index")
fig.colorbar(im0, ax=axes[0], orientation="vertical", fraction=0.015)
im1 = axes[1].imshow(event_pred.T, aspect="auto", interpolation="nearest", cmap="viridis")
axes[1].set_title("reconstructed masked event bytes; unmasked events are blank")
axes[1].set_ylabel("byte index")
fig.colorbar(im1, ax=axes[1], orientation="vertical", fraction=0.015)
im2 = axes[2].imshow(event_error.T, aspect="auto", interpolation="nearest", cmap="magma", vmin=0)
axes[2].set_title("absolute reconstruction error on masked event bytes; unmasked events are blank")
axes[2].set_xlabel("event index")
axes[2].set_ylabel("byte index")
fig.colorbar(im2, ax=axes[2], orientation="vertical", fraction=0.015)
plt.show()


In [ ]:
rows = []
for event_idx, pred_bytes in zip(masked_event_indices[sample].numpy(), event_bytes[sample]):
    event_idx = int(event_idx)
    for byte_idx, value in enumerate(pred_bytes):
        target = int(batch["events_uint8"][sample, event_idx, byte_idx])
        rows.append({"event_index": event_idx, "byte_index": byte_idx, "target_uint8": target, "pred_uint8": int(value), "abs_error": abs(int(value) - target)})
pl.DataFrame(rows).sort(["event_index", "byte_index"]).head(300)
